In [ ]:
import gradio as gr
import cv2
import os
import numpy as np
from PIL import Image
import sqlite3
import time

# Database Setup
db_path = "face_recognition.db"
dataset_path = "dataset"
trainer_path = "trainer"
os.makedirs(dataset_path, exist_ok=True)
os.makedirs(trainer_path, exist_ok=True)

# Create database and tables if they don't exist
def create_db():
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.execute('''CREATE TABLE IF NOT EXISTS users (
                        id INTEGER PRIMARY KEY AUTOINCREMENT,
                        name TEXT NOT NULL)''')
    cursor.execute('''CREATE TABLE IF NOT EXISTS images (
                        id INTEGER PRIMARY KEY AUTOINCREMENT,
                        user_id INTEGER,
                        image_path TEXT NOT NULL,
                        FOREIGN KEY(user_id) REFERENCES users(id))''')
    conn.commit()
    conn.close()

create_db()

# Global state to track if images are captured
images_captured = False

# Capture Training Data Function with Database Integration
def capture_training_data(user_name):
    global images_captured
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    
    # Insert user into database
    cursor.execute("INSERT INTO users (name) VALUES (?)", (user_name,))
    user_id = cursor.lastrowid
    conn.commit()

    cap = cv2.VideoCapture(0)
    face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
    sample_count = 0
    max_samples = 10
    instructions = ["Look straight", "Turn left", "Turn right", "Tilt up", "Tilt down"]
    instruction_interval = max_samples // len(instructions)
    current_instruction = 0

    while sample_count < max_samples:
        ret, frame = cap.read()
        if not ret:
            break
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        faces = face_cascade.detectMultiScale(gray, 1.3, 5)
        instruction_text = instructions[current_instruction]

        cv2.putText(frame, instruction_text, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)
        for (x, y, w, h) in faces:
            face_image = gray[y:y+h, x:x+w]
            image_path = f"{dataset_path}/User.{user_id}.{int(time.time())}_{sample_count}.jpg"
            cv2.imwrite(image_path, face_image)
            sample_count += 1
            
            # Insert image path into the database
            cursor.execute("INSERT INTO images (user_id, image_path) VALUES (?, ?)", (user_id, image_path))
            conn.commit()

            cv2.rectangle(frame, (x, y), (x+w, y+h), (255, 0, 0), 2)

        cv2.imshow("Capturing Training Data", frame)
        cv2.waitKey(1000) if sample_count % instruction_interval == 0 else cv2.waitKey(1)
        current_instruction = min(current_instruction + 1, len(instructions) - 1)

    cap.release()
    cv2.destroyAllWindows()
    images_captured = True  # Set the state to True once capturing is done
    conn.close()
    return f"Captured {sample_count} images for user '{user_name}'"

# Train Model Function with Database Integration
def train_model():
    if not images_captured:
        return "Please capture images first before training the model."

    recognizer = cv2.face.LBPHFaceRecognizer_create()
    face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # Retrieve all image paths and associated user IDs from the database
    cursor.execute('''SELECT images.image_path, users.id FROM images
                      JOIN users ON images.user_id = users.id''')
    rows = cursor.fetchall()
    
    faces = []
    ids = []

    for row in rows:
        image_path, user_id = row
        gray_image = Image.open(image_path).convert('L')
        image_np = np.array(gray_image, 'uint8')
        faces_detected = face_cascade.detectMultiScale(image_np)

        for (x, y, w, h) in faces_detected:
            faces.append(image_np[y:y+h, x:x+w])
            ids.append(user_id)

    recognizer.train(faces, np.array(ids))
    recognizer.write(f'{trainer_path}/trainer.yml')
    conn.close()
    return "Model trained successfully!"

# Recognize Faces Function with Database Integration
def recognize_faces():
    recognizer = cv2.face.LBPHFaceRecognizer_create()
    model_path = f'{trainer_path}/trainer.yml'

    if not os.path.exists(model_path):
        return "Model not found. Please train the model first."

    recognizer.read(model_path)
    face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
    cap = cv2.VideoCapture(0)
    confidence_threshold = 50

    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        faces = face_cascade.detectMultiScale(gray, 1.3, 5)

        for (x, y, w, h) in faces:
            id, confidence = recognizer.predict(gray[y:y+h, x:x+w])
            confidence_percentage = 100 - confidence
            cursor.execute("SELECT name FROM users WHERE id = ?", (id,))
            row = cursor.fetchone()
            name = row[0] if row else "Unknown"
            text = f"{name} ({confidence_percentage:.2f}%)"
            cv2.putText(frame, text, (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0) if name != "Unknown" else (0, 0, 255), 2)
            cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 255, 0) if name != "Unknown" else (0, 0, 255), 2)

        cv2.imshow("Face Recognition", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()
    conn.close()
    return "Face recognition ended."

# Gradio Interface Setup
def run_capture(user_name):
    return capture_training_data(user_name)

def run_training():
    return train_model()

def run_recognition():
    return recognize_faces()

with gr.Blocks(css=""" 
    html, body { background-color: #e0e0e0 !important; font-family: Arial, sans-serif; }
    .gradio-container { max-width: 500px; padding: 10px; box-shadow: 0 4px 12px rgba(0, 0, 0, 0.2); border-radius: 12px; background-color: #43A5BE; margin: 30px auto; }
    .gradio-tabs button { font-size: 14px; color: #444; padding: 10px 20px; }
    .gradio-button { background-color: #4CAF50; color: white; font-size: 18px; padding: 10px; border-radius: 5px; cursor: pointer; width: 100%; margin: 10px 0; }
    .gradio-textbox, .gradio-output { font-size: 14px; padding: 8px; margin: 8px 0; border: 1px solid #ccc; border-radius: 6px; width: 100%; background-color: #fff; }
    .title { text-align: center; font-size: 24px; font-weight: bold; color: #333; margin-bottom: 20px; }
""") as demo:
    gr.HTML('<div class="title">Face Recognition System</div>')

    with gr.Tab("New User Registration"):
        with gr.Column():
            name_input = gr.Textbox(label="Enter Name")
            capture_btn = gr.Button("Capture", elem_classes="gradio-button")
            capture_output = gr.Textbox(placeholder="Capture output will appear here...", elem_classes="gradio-output")
            capture_btn.click(fn=run_capture, inputs=name_input, outputs=capture_output)

    with gr.Tab("Train Model"):
        with gr.Column():
            train_btn = gr.Button("Train Model", elem_classes="gradio-button")
            train_output = gr.Textbox(placeholder="Training output will appear here...", elem_classes="gradio-output")
            train_btn.click(fn=run_training, outputs=train_output)

    with gr.Tab("Start Recognition"):
        with gr.Column():
            recognition_btn = gr.Button("Start Recognition", elem_classes="gradio-button")
            recognition_output = gr.Textbox(placeholder="Recognition output will appear here...", elem_classes="gradio-output")
            recognition_btn.click(fn=run_recognition, outputs=recognition_output)

demo.launch()
